# ⛏️ 第四课 · 亲手挖矿

## 本课目标

- 把前三课的知识拼成一个能跑的真矿工
- 亲手挖出一个区块，看到 nonce 从 0 涨到找到为止
- 理解矿工程序实际上有多简单

> 这一课的代码稍微长一点，但核心逻辑不超过 30 行。

## 1. 矿工的工作流程

一个矿工做的事情可以浓缩为这 7 步：

```
1. 从网络接收新区块模板（区块头 + 交易列表）
2. 插入 coinbase 交易（发工资给自己）
3. 计算 Merkle 根，填入区块头
4. nonce 从 0 开始
5. 序列化区块头 → 双 SHA-256 → 检查是否小于目标
6. 如果大于目标 → nonce++ → 回到第 5 步
7. 如果小于目标 → 挖到了！广播给全网
```

看起来很简单对吧？比特币矿工就是 7×24 小时重复第 5-6 步。

In [ ]:
import hashlib, struct, time, random
from dataclasses import dataclass, field
from typing import List, Optional

# ---- 重用之前的工具函数 ----
def sha256(data: bytes) -> bytes:
    return hashlib.sha256(data).digest()

def double_sha256(data: bytes) -> bytes:
    return hashlib.sha256(hashlib.sha256(data).digest()).digest()

def bits_to_target(bits: int) -> int:
    return (bits & 0x00ffffff) * (256 ** ((bits >> 24) - 3))

def target_to_difficulty(target: int) -> float:
    return bits_to_target(0x1d00ffff) / target

# 区块头定义（和第三课一样）
@dataclass
class BlockHeader:
    version: int = 1
    prev_hash: bytes = field(default_factory=lambda: b"\x00" * 32)
    merkle_root: bytes = field(default_factory=lambda: b"\x00" * 32)
    timestamp: int = 0
    bits: int = 0x1d00ffff
    nonce: int = 0

    def serialize(self) -> bytes:
        return struct.pack("<I32s32sIII",
            self.version, self.prev_hash, self.merkle_root,
            self.timestamp, self.bits, self.nonce)

    def hash(self) -> bytes:
        return double_sha256(self.serialize())[::-1]

def build_merkle_root(tx_data):
    hashes = [double_sha256(d) for d in tx_data]
    while len(hashes) > 1:
        if len(hashes) % 2 == 1:
            hashes.append(hashes[-1])
        hashes = [double_sha256(hashes[i] + hashes[i+1])
                  for i in range(0, len(hashes), 2)]
    return hashes[0] if hashes else b"\x00" * 32

print("✅ 工具函数准备就绪")  

## 2. 核心挖矿循环

下面这个 `Miner` 类就是完整的矿工。核心就是一个 for 循环 + 一个 if 判断。

```python
for nonce in range(最大值):
    header.nonce = nonce
    hash = double_sha256(header.serialize())
    if hash < target:
        return nonce  # 挖到了！
# 没找到，改 extraNonce 或时间戳继续
```

就这么简单。矿工不需要懂区块链、不需要懂经济学。只需要不停地算哈希。

In [ ]:
class Miner:
    # 模拟一个比特币矿工
    def __init__(self, name: str, hashrate: int = 5_000_000):
        self.name = name
        self.hashrate = hashrate  # 模拟算力 (哈希/秒)
        self.total_hashes = 0
        self.blocks_found = 0

    def mine(self, header: BlockHeader, max_seconds: float = 30.0) -> Optional[int]:
        # 核心挖矿循环
        target = bits_to_target(header.bits)
        start = time.time()
        last_report = start

        print(f"\n⛏️  {self.name} 开始挖矿")
        print(f"    目标:   < 0x{target:064x}")
        print(f"    难度:   {target_to_difficulty(target):,.0f}")
        print(f"    算力:   {self.hashrate:,} H/s (模拟)")
        print()

        for nonce in range(2**32):  # 遍历 nonce 空间
            header.nonce = nonce
            h_int = int.from_bytes(double_sha256(header.serialize()), "big")
            self.total_hashes += 1

            if h_int < target:
                # 找到了！
                elapsed = time.time() - start
                self.blocks_found += 1
                print(f"\n🎉 挖到区块！！！")
                print(f"    Nonce:   {nonce:,}")
                print(f"    哈希:    {header.hash().hex()}")
                print(f"    耗时:    {elapsed:.2f} 秒")
                print(f"    实际算力: {self.total_hashes/elapsed:,.0f} H/s")
                return nonce

            # 每秒报告一次进度
            now = time.time()
            if now - last_report >= 1:
                elapsed = now - start
                hps = self.total_hashes / elapsed if elapsed > 0 else 0
                print(f"    nonce={nonce:>12,} | {self.total_hashes:>12,} hashes | "
                      f"{elapsed:>5.0f}s | {hps:,.0f} H/s", end="\r")
                last_report = now

            if time.time() - start > max_seconds:
                print(f"\n⏰ 超时 ({max_seconds}s)，未找到区块")
                print(f"    共尝试: {self.total_hashes:,} 次")
                return None

        return None

print("✅ Miner 类定义完成")  

## 3. 运行！亲眼见证一次挖矿

下面我们会创建一个区块模板，然后让矿工去找 nonce。
因为用的是 Python（不是 C/ASIC），我们设了很低的难度，
让你能在几秒内看到结果。

In [ ]:
# 准备区块内容
# 1. 创建一些模拟交易
print("📝 准备区块模板...")
tx_data = []
for i in range(5):
    tx_data.append(f"Alice pays Bob {random.randint(1, 50)} BTC".encode())

# 2. 加 coinbase 交易
coinbase = f"COINBASE -> Miner 奖励 3.125 BTC".encode()
all_txs = [coinbase] + tx_data

# 3. 计算 Merkle 根
merkle = build_merkle_root(all_txs)

# 4. 构造区块头
header = BlockHeader(
    prev_hash=bytes.fromhex("0" * 64),
    merkle_root=merkle,
    timestamp=int(time.time()),
    bits=0x2000ffff,  # 比真实比特币低很多，让你能挖到
)

# 5. 开始挖矿
miner = Miner(name="龙虾矿工 🦞", hashrate=10_000_000)
result = miner.mine(header, max_seconds=30)

if result is not None:
    print(f"\n{'='*60}")
    print(f"  🎉 挖矿成功！")
    print(f"  区块哈希: {header.hash().hex()}")
    print(f"  Nonce:    {result}")
    print(f"  包含交易: {len(all_txs)} 笔")
    print(f"{'='*60}")
else:
    print("\n❌ 这次没挖到，正常现象！")  

## 4. 算力对比：为什么 Python 挖不了真比特币

上面我们挖到了，是因为难度设得很低。如果设成真实的比特币难度呢？

来看不同设备在真实难度下的预期挖矿时间：

In [ ]:
def expected_time_for_target(target: int, hashrate: float) -> float:
    return (2**256 / target) / hashrate

target = bits_to_target(0x2000ffff)  # 我们刚才用的演示难度
real_target = difficulty_to_target(70_000_000_000_000)  # 真实难度

def fmt(s):
    if s < 60:      return f"{s:.1f} 秒"
    if s < 3600:    return f"{s/60:.0f} 分钟"
    if s < 86400:   return f"{s/3600:.1f} 小时"
    if s < 3.15e7:  return f"{s/86400:.0f} 天"
    return f"{s/3.15e7:,.0f} 年"

print(f"{'设备':<20} | {'演示难度':>10} | {'真实难度':>16}")
print("-" * 52)
for name, hr in [
    ("Python 模拟", 10_000_000),
    ("C 语言实现", 50_000_000),
    ("GPU (CUDA)", 3_000_000_000),
    ("Antminer S21", 200_000_000_000_000),
]:
    t_demo = expected_time_for_target(target, hr)
    t_real = expected_time_for_target(real_target, hr)
    print(f"{name:<20} | {fmt(t_demo):>10} | {fmt(t_real):>16}")

print()
print("💀 Python 挖真实比特币 ≈ 宇宙年龄的数十亿倍")
print("💡 ASIC 矿机在真实难度下约 10 分钟出块 —— 这就是设计目标")  

## 本课小结

| 要点 | 记住 |
|------|------|
| 矿工循环 | for nonce: hash = sha256d(header); if hash < target: 挖到了 |
| 核心简单 | 整个矿工核心约 30 行代码 |
| 难度决定一切 | 演示难度 vs 真实难度差 70 万亿倍 |
| Python 挖不了 | 不是不能，是太慢——差了几十亿倍 |

**下一课**：矿池——一个人挖是买彩票，一群人挖是开工厂。